# Temperature and Variance [Step 03.01]

> **MLCourse - Agentic AI - Agent Patterns**

Every technique in this module - self-consistency, tree search, best-of-n -
depends on one thing being true:

> Asking the same question twice can give you two different answers.

That variance is not a defect to be tuned away. It is the **raw material**. This
notebook measures it before we start spending money on it.

```
   temperature 0.0        temperature 0.7        temperature 1.2
   ----------------       ----------------       ----------------
   A A A A A              A A B A A              A B C A D
   no variance            some variance          lots of variance
   nothing to vote on     votes are meaningful   votes are noise
```

### What you'll learn

- What temperature actually does to the token distribution.
- Measured output diversity at three temperatures, on the same prompt.
- The crucial asymmetry: variance helps on **reasoning paths**, hurts on
  **formatting**.
- Why `temperature=0` is not actually deterministic in practice.

### Why it matters

If you sample five times at temperature 0 you get five near-identical answers,
pay five times, and learn nothing. If you sample five times at temperature 1.5
you get five unrelated answers and a majority vote of one. The techniques in
notebooks 02-04 only work inside a temperature band, and this notebook finds it.

### Prerequisites

- [01_langchain/01_fundamentals](../../../01_langchain/01_fundamentals) - model parameters.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### 1. What temperature does

The model produces a score (a *logit*) for every token in its vocabulary.
Temperature divides those scores before they are turned into probabilities:

```
   p(token) = softmax(logits / T)
```

- **T -> 0**: the division blows up the differences, the top token's probability
  approaches 1. Greedy decoding.
- **T = 1**: the model's own distribution, untouched.
- **T > 1**: differences shrink, unlikely tokens get a real chance.

The consequence that matters here: **low temperature does not mean "more
correct"**. It means "more committed to the model's first instinct". If that
instinct is wrong, low temperature makes it wrong every single time - and no
amount of resampling will rescue you.

### The task set


In [ ]:
# Six multi-step word problems with short, exactly checkable answers. They are
# deliberately the kind that a model usually gets right but sometimes slips on:
# several dependent steps, one trap each (a percentage of the WRONG base, a
# time carry, an off-by-one week, a compounding discount).
#
# Small on purpose. Every technique in this module multiplies the number of
# calls, and the Groq free tier gives us 8000 tokens per minute.

TASKS = [
    dict(id="tank",
         q="A tank holds 480 litres and is currently 3/8 full. You add 90 litres, "
           "then drain an amount equal to 15% of the tank's TOTAL CAPACITY. "
           "How many litres are in the tank now?",
         answer="198"),
    dict(id="train",
         q="A train leaves at 09:40 and travels for 2 hours 50 minutes. You then "
           "wait 35 minutes for a connection, then travel a further 1 hour 15 "
           "minutes. At what time do you arrive? Use 24-hour HH:MM format.",
         answer="14:20"),
    dict(id="book",
         q="A book has 250 pages. Ana reads 12 pages per day on weekdays and 30 "
           "pages per day at weekends. She starts on a Monday morning. On which "
           "day of the week does she finish the book?",
         answer="monday"),
    dict(id="discount",
         q="A jacket costs 200 EUR. A 20% discount is applied, and then a further "
           "15% is taken off the already-reduced price. What is the final price "
           "in EUR?",
         answer="136"),
    dict(id="rect",
         q="A rectangle is three times as long as it is wide. Its perimeter is "
           "64 cm. What is its area in square centimetres?",
         answer="192"),
    dict(id="coins",
         q="You have 17 coins, all of them either 5 cents or 20 cents, worth 205 "
           "cents in total. How many 20-cent coins are there?",
         answer="8"),
]

ANSWER_INSTRUCTION = ("Work through it step by step, briefly. Then give your final "
                      "answer on the last line in exactly this form:\nANSWER: <value>")

import re


def parse_answer(text: str) -> str:
    """Pull the value out of the last 'ANSWER:' line. Returns '' if absent."""
    matches = re.findall(r"ANSWER\s*:\s*(.+)", text, re.I)
    return matches[-1].strip() if matches else ""


def normalise(value: str) -> str:
    """Make answers comparable: lowercase, drop units, currency and separators."""
    v = value.strip().lower()
    v = re.sub(r"\*\*|`|\.$", "", v)
    v = re.sub(r"\b(litres?|liters?|eur|euros?|cm|square centimetres?|cm\^?2|"
               r"cents?|coins?|pages?|hours?)\b", "", v)
    v = v.replace(",", "").replace(" ", "")
    return v.strip()


def is_correct(given: str, expected: str) -> bool:
    g, e = normalise(given), normalise(expected)
    if not g:
        return False
    return g == e or g.endswith(e) or e in g.split("=")[-1]


print("%d tasks" % len(TASKS))
for t in TASKS:
    print("  %-9s expected %-8s | %s..." % (t["id"], t["answer"], t["q"][:56]))


### 2. Measuring diversity

We take one task, sample it five times at each of three temperatures, and
measure two things:

- **Distinct final answers** - how much the *conclusion* varies.
- **Distinct reasoning texts** - how much the *path* varies.

Those two can move independently, and the gap between them is the whole
opportunity.

In [3]:
TASK = TASKS[0]          # the tank problem
N = 5
TEMPS = [0.0, 0.7, 1.2]

runs = {}
for T in TEMPS:
    samples = []
    for i in range(N):
        out = chat([("user", TASK["q"] + "\n\n" + ANSWER_INSTRUCTION)],
                   temperature=T, max_tokens=320)
        samples.append(out.content.strip())
    runs[T] = samples
    answers = [parse_answer(s) for s in samples]
    print("T=%.1f  answers: %s" % (T, [a[:10] for a in answers]))

T=0.0  answers: ['198', '198', '198', '198', '198']


T=0.7  answers: ['', '198', '198', '', '198']


T=1.2  answers: ['198', '1', '198', '198', '198']


In [4]:
print("%6s %12s %12s %10s %10s" % ("temp", "distinct", "distinct", "correct", "mean out"))
print("%6s %12s %12s %10s %10s" % ("", "answers", "reasonings", "", "tokens"))
print("-" * 56)
for T in TEMPS:
    samples = runs[T]
    answers = [normalise(parse_answer(s)) for s in samples]
    correct = sum(is_correct(parse_answer(s), TASK["answer"]) for s in samples)
    print("%6.1f %12d %12d %9d/%d %10d"
          % (T, len(set(answers)), len(set(samples)), correct, N,
             sum(approx_tokens(s) for s in samples) / N))

  temp     distinct     distinct    correct   mean out
            answers   reasonings                tokens
--------------------------------------------------------
   0.0            1            1         5/5        184
   0.7            2            5         3/5        264
   1.2            2            5         4/5        228


### The asymmetry to notice

Look at the two "distinct" columns. Even at temperature 0, the *reasoning* texts
are often not all identical - and at moderate temperature the reasoning varies
much more than the answer does.

That gap is the opportunity self-consistency exploits:

> Different reasoning paths, converging on the same answer, is **evidence**.
> Different reasoning paths, diverging, is a **warning**.

Neither signal exists at temperature 0, because there is only one path.

### Show the first two sampled reasonings at T=0.7 side by side, truncated.


In [ ]:
a, b = runs[0.7][0], runs[0.7][1]
print("SAMPLE 1:\n%s\n" % a[:420])
print("-" * 70)
print("SAMPLE 2:\n%s" % b[:420])


### 3. "Temperature 0 is deterministic" is not quite true

Two reasons you will see variation even at `temperature=0`:

1. **Floating-point non-determinism on the server.** Batched GPU inference does
   not guarantee identical numerics across runs, so ties break differently.
2. **You do not control the whole stack.** Model versions, quantisation and
   routing change under you without notice.

So `temperature=0` gives you *low* variance, not *no* variance. Design for
"usually the same", never for "byte-identical" - and if you truly need
reproducibility, cache the responses rather than trusting the parameter.

In [6]:
identical = len(set(runs[0.0])) == 1
print("all %d samples at T=0.0 byte-identical: %s" % (N, identical))
if not identical:
    print("-> %d distinct texts. This is normal and is why you cache, not hope."
          % len(set(runs[0.0])))

all 5 samples at T=0.0 byte-identical: True


### 4. Where variance helps and where it hurts

This is the practical rule, and getting it backwards is the most common mistake:

| Use LOW temperature for | Use MODERATE temperature for |
|---|---|
| Structured output (JSON, code) | Reasoning chains you intend to vote on |
| Tool call arguments | Brainstorming candidate plans |
| Classification and routing | Anything sampled more than once |
| Anything parsed by a regex | Creative or open-ended writing |

Note the trap: **an agent does both in the same request.** It reasons (wants
variance) and then emits a tool call (wants none). The way out is to split the
call - sample the reasoning warm, then formalise the chosen answer at
temperature 0.

### Demonstrate the format-fragility half of that rule.


In [ ]:
FORMAT_TASK = ('Return the tank problem\'s answer as JSON with exactly these keys: '
               '{"litres": <number>, "steps": <integer count>}. JSON only, no prose.\n\n'
               + TASK["q"])

import json as _json

for T in (0.0, 1.2):
    ok = 0
    for _ in range(3):
        txt = chat([("user", FORMAT_TASK)], temperature=T, max_tokens=140).content.strip()
        txt = txt.strip("`").replace("json\n", "", 1)
        try:
            _json.loads(txt)
            ok += 1
        except Exception:
            pass
    print("T=%.1f  parsed as valid JSON: %d/3" % (T, ok))


### 5. Pitfalls

- **Turning temperature up to "make it smarter".** It makes it more varied. Those
  are different, and only the sampling techniques in this module convert one into
  the other.
- **Sampling at temperature 0.** You pay N times for one answer.
- **Sampling above ~1.2 for reasoning.** The votes become noise and the majority
  is meaningless.
- **One temperature for the whole agent.** Reasoning and formatting want
  opposite settings.
- **Assuming reproducibility.** Cache if you need it.

### Recap

| Idea | Takeaway |
|---|---|
| Temperature scales logits | Low = committed to the first instinct, not "correct" |
| Variance is the raw material | No variance, nothing to vote on |
| Paths vary more than answers | That gap is what self-consistency exploits |
| Low for format, moderate for reasoning | Split the call if you need both |
| T=0 is not deterministic | Cache instead of trusting it |

**Next:** [02_self_consistency](02_self_consistency.ipynb) - turn that variance
into accuracy, and measure whether it actually worked.